# Pipeline PLN — EDA e Ingeniería de Características con Normalización de Sinónimos

**Proyecto:** Análisis de Corpus de Encuestas Universitarias
**Versión:** `beta_2`
**Fases cubiertas:** Fase 1 (limpieza + sinónimos) · Fase 2 (EDA) · Fase 3 (Ingeniería)

---

## Arquitectura del pipeline

```
                    ┌──────────────────────────────────────────────┐
                    │            settings.py                        │
                    │  (fuente única: accents, stopwords,           │
                    │   canonical_synonyms_seeds, colloquialisms)   │
                    └──────────────────────────────────────────────┘
                             │              │                │
              ┌──────────────┘              │                └───────────────┐
              ▼                             ▼                                ▼
     ┌─────────────────┐         ┌────────────────────┐          ┌────────────────────┐
     │   limpieza.py   │         │    synonyms.py     │          │   (config global)  │
     │    Cleaner      │         │  SynonymReplacer   │          └────────────────────┘
     └─────────────────┘         └────────────────────┘
              │                             │
    clean_key_column()            build_synonyms_dict()
              │                             │
     ► columna _clean                       │
              │                             │
    eliminate_stopwords()                   │
              │                             │
     ► columna _no_stopwords ──────► synonyms_to_dataframe()
                                            │
                                   ► columna _no_stopwords_synonyms
                                            │
                          ┌─────────────────┴─────────────────┐
                          ▼                                    ▼
                   ┌─────────────┐                    ┌──────────────────┐
                   │   EDA.py    │                    │  Engineering.py  │
                   │ nubes,      │                    │  BoW, TF-IDF,    │
                   │ n-gramas,   │                    │  reducción de    │
                   │ longitudes  │                    │  vocabulario     │
                   └─────────────┘                    └──────────────────┘
```

## Decisión metodológica (integración de la PR de Julián)

La rama de Julián aporta `synonyms.py`, que colapsa variantes léxicas
sobre un **término canónico** usando embeddings Word2Vec en español más
un diccionario curado de colombianismos. Tras el *merge*, el orden
canónico de columnas del pipeline es:

| Columna | Generada por | Contenido |
|---|---|---|
| `_clean` | `Cleaner.clean_key_column()` | Texto normalizado (minúsculas, sin tildes ni símbolos) |
| `_no_stopwords` | `Cleaner.eliminate_stopwords()` | Sin palabras vacías |
| `_no_stopwords_synonyms` | `SynonymReplacer.synonyms_to_dataframe()` | Variantes colapsadas a término canónico |

**Todas las nubes de palabras y n-gramas de este cuaderno se calculan
sobre la columna `_no_stopwords_synonyms`.** Esto reduce el vocabulario,
concentra la señal y produce nubes más legibles para la presentación
final.

## Paso 1 — Importación de módulos y configuración global

In [1]:
%pip install gensim --quiet
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")
%matplotlib inline

# ── Robust path resolution ────────────────────────────────────────────────────
# Walks UP from cwd until pyproject.toml is found (project root).

def _find_project_root(marker: str = "pyproject.toml") -> Path:
    """Walk up from cwd until a directory containing *marker* is found."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find '{marker}' in any parent of {Path.cwd().resolve()}.\n"
        "Make sure pyproject.toml exists at the project root."
    )

_PROJECT_ROOT     = _find_project_root()
_SRC_DATACLEANING = _PROJECT_ROOT / "src" / "datacleaning"  # limpieza, synonyms
_SRC              = _PROJECT_ROOT / "src"                    # EDA, Engineering, settings
_DATA             = _PROJECT_ROOT / "data" / "utilities"

# ── Add both src paths to sys.path ───────────────────────────────────────────
for _path in (_SRC_DATACLEANING, _SRC):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

# ── Validate that all required modules are reachable ─────────────────────────
_module_locations = {
    "limpieza"   : _SRC_DATACLEANING,
    "synonyms"   : _SRC_DATACLEANING,
    "settings"   : _SRC,
    "EDA"        : _SRC,
    "Engineering": _SRC,
}
_missing = [
    f"{m}.py  (expected in {loc})"
    for m, loc in _module_locations.items()
    if not (loc / f"{m}.py").exists()
]
if _missing:
    raise FileNotFoundError(
        "\nThe following modules were not found:\n"
        + "\n".join(f"  ✗ {entry}" for entry in _missing)
    )

# ── Locate the corpus automatically ──────────────────────────────────────────
# Searches for any Excel file inside data/utilities/, excluding the auxiliary
# accents/stopwords spreadsheets. Uses os.listdir (sees everything, including
# OneDrive-synced and git-ignored files) rather than relying on the exact name.
def _find_corpus(data_dir: Path) -> Path:
    """Return the survey-corpus Excel file inside *data_dir*.

    Auxiliary spreadsheets (accents, stopwords) and Excel lock files
    (~$...) are ignored. The largest remaining Excel file is taken as
    the corpus. The search checks the top-level directory first, then
    falls back to a recursive search if necessary.
    """
    if not data_dir.exists():
        raise FileNotFoundError(f"Data directory not found: {data_dir}")

    auxiliary = {"accents", "stopwords"}
    excel_ext = {".xlsx", ".xls", ".xlsm"}

    def eligible(path: Path) -> bool:
        if not path.is_file():
            return False
        if path.suffix.lower() not in excel_ext:
            return False
        if path.stem.lower() in auxiliary:
            return False
        if path.name.startswith("~$"):  # Excel lock file
            return False
        return True

    candidates = [path for path in data_dir.iterdir() if eligible(path)]
    if not candidates:
        candidates = [path for path in data_dir.rglob("*") if eligible(path)]

    if not candidates:
        present = [repr(p.name) for p in data_dir.iterdir()]
        raise FileNotFoundError(
            f"No corpus Excel file found in {data_dir}.\n"
            f"Files actually present:\n  "
            + "\n  ".join(present)
            + "\n\nIf the corpus appears here but was skipped, check that "
            "its extension is .xlsx/.xls/.xlsm and its name is not "
            "'accents' or 'stopwords'."
        )

    return max(candidates, key=lambda p: p.stat().st_size)

# Robust corpus resolution with helpful fallbacks
try:
    EXCEL_PATH = _find_corpus(_DATA)
except FileNotFoundError:
    # 1) Try common explicit filename(s)
    fallbacks = list(_DATA.glob("Corpus*.xls*")) + list(_DATA.glob("*Corpus*.xls*"))
    if not fallbacks:
        # 2) Try searching the whole data tree (in case file lives in a sibling dir)
        fallbacks = list((_PROJECT_ROOT / "data").rglob("Corpus*.xls*")) if (_PROJECT_ROOT / "data").exists() else []

    if fallbacks:
        EXCEL_PATH = max(fallbacks, key=lambda p: p.stat().st_size)
    else:
        # Re-raise with the original, more informative message
        raise FileNotFoundError(
            f"No corpus Excel file found in {_DATA}.\n"
            f"Checked for common fallbacks (e.g. 'Corpus*.xlsx') and recursive search under {_PROJECT_ROOT / 'data'}.\n"
            "Make sure the corpus Excel file is placed in data/utilities/ and its name is not 'accents' or 'stopwords'."
        )

print(f"\u2705 Project root : {_PROJECT_ROOT}")
print(f"\u2705 src/datacleaning + src/ added to path")
print(f"\u2705 Corpus       : {EXCEL_PATH.name}  ({EXCEL_PATH.stat().st_size / 1e6:.1f} MB)")

# Phase 1 — cleaning and synonym normalisation
from limpieza import Cleaner
import settings

SYNONYMS_MODEL_PATH = _PROJECT_ROOT / "models" / "word2vec_model.bin"
settings.SYNONYMS_MODEL_PATH = SYNONYMS_MODEL_PATH

from synonyms import SynonymReplacer

# Phase 2 — EDA
from EDA import (
    NgramAnalyzer,
    SurveyWordClouds,
    TextLengthAnalyzer,
    WordCloudGenerator,
    run_full_eda,
)

# Phase 3 — Feature engineering
from Engineering import (
    BagOfNgrams,
    TfIdfTransformer,
    build_feature_matrices,
    compare_vocabularies,
    top_collapsed_terms,
    vocabulary_reduction,
)

print("\u2705 All pipeline modules imported successfully.")



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ Project root : C:\Users\Acer\OneDrive\Escritorio\MINE009_Externado\CAPSTONE_U_EXTERNADO\capstonenlpencuestas_mine9
✅ src/datacleaning + src/ added to path
✅ Corpus       : Corpus Ejemplo PLN.xlsx  (12.2 MB)


ImportError: cannot import name 'SYNONYMS_MODEL_URL' from 'settings' (C:\Users\Acer\OneDrive\Escritorio\MINE009_Externado\CAPSTONE_U_EXTERNADO\capstonenlpencuestas_mine9\src\settings.py)

### 1.1 Parámetros globales del corpus

Cada hoja del Excel es una **encuesta** distinta con sus propias
columnas de texto abierto. El diccionario `SURVEY_CONFIG` es la única
fuente de verdad sobre qué columnas analizar por encuesta.

In [ ]:

# Directory where PNGs for the final PowerPoint will be exported
FIGURES_DIR = _PROJECT_ROOT / "figures"
WORDCLOUDS_DIR = FIGURES_DIR / "wordclouds"
NGRAMS_DIR = FIGURES_DIR / "ngrams"
for _d in (WORDCLOUDS_DIR, NGRAMS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# Stopwords workbook (globales + particulares sheets).
# Passed to Cleaner so "nr", "facultad", etc. are removed.
STOPWORDS_PATH = _DATA / "stopwords.xlsx"
assert STOPWORDS_PATH.exists(), f"stopwords.xlsx not found at {STOPWORDS_PATH}"

# One primary open-text column per survey (the richest one)
SURVEY_CONFIG: dict[str, str] = {
    "Evaluaci\u00f3n Docente": (
        "Observaciones - 19. Aspectos para resaltar del docente "
        "y/o del espacio acad\u00e9mico"
    ),
    "Autoevaluaci\u00f3n Docente": (
        "Aspectos para resaltar de su labor como docente y/o del "
        "espacio acad\u00e9mico"
    ),
    "Calidad Docentes": (
        "Si desea complementar sus respuestas o hacer sugerencias "
        "para nuestro mejoramiento continuo, por favor incl\u00fayalas "
        "a continuaci\u00f3n (si no tiene comentarios, dejar este "
        "espacio en blanco):"
    ),
    "Calidad Administrativos": (
        "Si desea complementar sus respuestas o hacer sugerencias "
        "para nuestro mejoramiento continuo, por favor incl\u00fayalas "
        "a continuaci\u00f3n (si no tiene comentarios, dejar este "
        "espacio en blanco):"
    ),
    "Calidad Estudiantes": (
        "Si desea complementar sus respuestas o hacer sugerencias "
        "para nuestro mejoramiento continuo, por favor incl\u00fayalas "
        "a continuaci\u00f3n (si no tiene comentarios, dejar este "
        "espacio en blanco):"
    ),
}

print(f"Surveys configured: {len(SURVEY_CONFIG)}")
for survey, column in SURVEY_CONFIG.items():
    print(f"  \u2022 {survey}")

## Paso 2 — Construcción del diccionario de sinónimos

`SynonymReplacer` descarga (una sola vez) el modelo Word2Vec en español
y construye el diccionario `{variante: canónico}` combinando:

1. Las semillas canónicas de `settings.canonical_synonyms_seeds`.
2. Los vecinos más cercanos de cada semilla según Word2Vec, filtrados
   por el umbral de similitud coseno.
3. El diccionario curado de colombianismos (que tiene prioridad).

> **Nota:** la primera ejecución descarga ~1 GB del modelo. Las
> siguientes lo cargan desde disco. El diccionario resultante se reutiliza
> para **todas** las encuestas — se construye una sola vez.

In [ ]:
# Build the synonym dictionary once, reuse across all surveys
replacer = SynonymReplacer()
replacer.load_model()
synonyms_dict = replacer.build_synonyms_dict()

print(f"Synonym dictionary size: {len(synonyms_dict)} mappings\n")
print("Sample {variant -> canonical} mappings:")
for variant, canonical in list(synonyms_dict.items())[:15]:
    marker = "  (identity)" if variant == canonical else ""
    print(f"  {variant:>18} \u2192 {canonical}{marker}")

## Paso 3 — Carga, limpieza y normalización de sinónimos (todas las encuestas)

Para cada encuesta se ejecuta el pipeline completo de Fase 1:
`clean_key_column()` → `eliminate_stopwords()` → `synonyms_to_dataframe()`.
El resultado se almacena en `corpus_by_survey`.

In [ ]:
def process_survey(survey_name: str, key_column: str) -> pd.DataFrame:
    """Run the full Phase-1 pipeline for a single survey.

    Args:
        survey_name: Excel sheet name (also the survey label).
        key_column: Open-text column to process.

    Returns:
        DataFrame with _clean, _no_stopwords and
        _no_stopwords_synonyms columns.
    """
    cleaner = Cleaner(
        file_path=EXCEL_PATH,
        survey_name=survey_name,
        key_column=key_column,
        sheet_name=survey_name,
        stopwords_path=STOPWORDS_PATH,
    )
    cleaner.clean_key_column()
    frame = cleaner.eliminate_stopwords()

    nosw_column = f"{key_column}_no_stopwords"
    frame = replacer.synonyms_to_dataframe(frame, nosw_column)
    return frame


corpus_by_survey: dict[str, pd.DataFrame] = {}

for survey_name, key_column in SURVEY_CONFIG.items():
    print(f"\u2500\u2500 Processing: {survey_name} \u2500\u2500")
    frame = process_survey(survey_name, key_column)
    corpus_by_survey[survey_name] = frame

    syn_col = f"{key_column}_no_stopwords_synonyms"
    non_empty = frame[syn_col].replace("", pd.NA).dropna().shape[0]
    print(f"   rows={frame.shape[0]:>6} | non-empty responses={non_empty}")

print("\n\u2705 All surveys processed through clean \u2192 stopwords \u2192 synonyms.")

In [ ]:
# Verify the column contract for one survey
demo_survey = "Evaluaci\u00f3n Docente"
demo_col = SURVEY_CONFIG[demo_survey]
demo_df = corpus_by_survey[demo_survey]

pipeline_cols = [
    demo_col,
    f"{demo_col}_clean",
    f"{demo_col}_no_stopwords",
    f"{demo_col}_no_stopwords_synonyms",
]

# Show a row where synonyms actually changed something
mask = (
    demo_df[f"{demo_col}_no_stopwords"]
    != demo_df[f"{demo_col}_no_stopwords_synonyms"]
)
sample = demo_df[mask].iloc[0]

print(f"Pipeline transformation for one response ({demo_survey}):\n")
labels = ["RAW", "_clean", "_no_stopwords", "_synonyms"]
for label, col in zip(labels, pipeline_cols):
    print(f"  [{label:>13}] {str(sample[col])[:85]}")

## Paso 4 — Impacto cuantitativo de la normalización de sinónimos

Antes de visualizar, medimos **cuánto** reduce el vocabulario la
sustitución por sinónimos. Esta es la evidencia analítica de que el
merge con la rama de Julián mejora las nubes y los n-gramas: menos
términos redundantes, señal más concentrada.

In [ ]:
impact_rows = []

for survey_name, frame in corpus_by_survey.items():
    key_column = SURVEY_CONFIG[survey_name]
    before = frame[f"{key_column}_no_stopwords"]
    after = frame[f"{key_column}_no_stopwords_synonyms"]

    reduction = vocabulary_reduction(before, after)
    reduction.insert(0, "Survey", survey_name)
    impact_rows.append(reduction)

impact_summary = pd.concat(impact_rows, ignore_index=True)
print("Vocabulary reduction by survey:\n")
display(impact_summary)

In [ ]:
# Which specific terms were collapsed in the main survey?
before = demo_df[f"{demo_col}_no_stopwords"]
after = demo_df[f"{demo_col}_no_stopwords_synonyms"]

print(f"Top terms collapsed into canonical forms ({demo_survey}):\n")
display(top_collapsed_terms(before, after, top_k=15))

## Paso 5 — Nubes de palabras por encuesta (entregable para la PPT)

Esta es la sección central del cuaderno. Se genera **una nube de palabras
por encuesta**, calculada sobre la columna `_no_stopwords_synonyms`, y se
exporta como PNG de alta resolución a `figures/wordclouds/`.

Cada PNG está listo para insertarse en el espacio reservado de la
presentación final. El nombre de archivo identifica inequívocamente a
qué encuesta corresponde.

In [ ]:
# SurveyWordClouds resolves the _synonyms column automatically per survey
survey_clouds = SurveyWordClouds(
    corpus_by_survey=corpus_by_survey,
    column_resolver="_no_stopwords_synonyms",
    colormap="viridis",
)

cloud_figures = survey_clouds.generate_all(
    save_dir=str(WORDCLOUDS_DIR),
    figsize=(13, 6),
    show_plots=True,
)

print(f"\n\u2705 {len(cloud_figures)} word clouds exported to {WORDCLOUDS_DIR}/")
print("\nPNG files ready for the PowerPoint:")
for png in sorted(WORDCLOUDS_DIR.glob("*.png")):
    print(f"  \u2022 {png.name}")

### 5.1 Mapa de nubes → diapositivas

| Encuesta | Archivo PNG | Diapositiva PPT |
|---|---|---|
| Evaluación Docente | `wordcloud_evaluacion_docente.png` | *(espacio reservado)* |
| Autoevaluación Docente | `wordcloud_autoevaluacion_docente.png` | *(espacio reservado)* |
| Calidad Docentes | `wordcloud_calidad_docentes.png` | *(espacio reservado)* |
| Calidad Administrativos | `wordcloud_calidad_administrativos.png` | *(espacio reservado)* |
| Calidad Estudiantes | `wordcloud_calidad_estudiantes.png` | *(espacio reservado)* |

> Los nombres de archivo se generan automáticamente por *slug* del nombre
> de la encuesta, de modo que cada nube queda identificada sin ambigüedad.

## Paso 6 — N-Gramas con normalización de sinónimos

Los n-gramas se recalculan sobre la columna `_no_stopwords_synonyms`.
Al colapsar variantes, los bigramas y trigramas se consolidan
(p. ej. *"buen docente"* y *"excelente profesor"* convergen hacia
*"bueno profesor"*), reforzando los patrones dominantes.

In [ ]:
def survey_ngrams(survey_name: str, top_n: int = 12) -> None:
    """Print top uni/bi/tri-grams for a survey's synonym column.

    Args:
        survey_name: Survey to analyse.
        top_n: Number of rows to display per n-gram size.
    """
    key_column = SURVEY_CONFIG[survey_name]
    syn_col = f"{key_column}_no_stopwords_synonyms"
    analyzer = NgramAnalyzer(corpus_by_survey[survey_name][syn_col])

    print(f"\n{'=' * 60}")
    print(f"  {survey_name}")
    print(f"{'=' * 60}")
    for n, label in ((1, "Unigrams"), (2, "Bigrams"), (3, "Trigrams")):
        table = analyzer.frequency_table(n).head(top_n)
        print(f"\n  Top {top_n} {label}:")
        print(table.to_string(index=False))
    return analyzer


# Detailed n-gram tables for the main survey
main_analyzer = survey_ngrams(demo_survey, top_n=12)

In [ ]:
# Comparative n-gram charts for the main survey (exported for PPT)
for n in (1, 2, 3):
    fig = main_analyzer.plot_comparative_ngrams(n=n, tops=(20, 30))
    label = {1: "unigrams", 2: "bigrams", 3: "trigrams"}[n]
    out_path = NGRAMS_DIR / f"ngrams_{label}_evaluacion_docente.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"\u2713 Saved: {out_path}")

In [ ]:
# Unigram bar charts for ALL surveys (compact, one per survey)
for survey_name in SURVEY_CONFIG:
    key_column = SURVEY_CONFIG[survey_name]
    syn_col = f"{key_column}_no_stopwords_synonyms"
    analyzer = NgramAnalyzer(corpus_by_survey[survey_name][syn_col])

    from EDA import _slugify  # filesystem-safe slug helper
    out_path = NGRAMS_DIR / f"top_unigrams_{_slugify(survey_name)}.png"
    fig = analyzer.plot_top_ngrams(
        n=1,
        top_k=20,
        title=f"Top 20 unigramas \u2014 {survey_name}",
        save_path=str(out_path),
    )
    plt.show()
    print(f"\u2713 {survey_name}: {out_path.name}")

## Paso 7 — Distribución de longitud de respuestas

Estadísticos y visualizaciones de longitud (caracteres y palabras) sobre
la columna normalizada, para la encuesta principal.

In [ ]:
length_analyzer = TextLengthAnalyzer(
    demo_df[f"{demo_col}_no_stopwords_synonyms"],
    column_name="Evaluaci\u00f3n Docente (synonyms)",
)
display(length_analyzer.summary())

In [ ]:
fig = length_analyzer.plot_histograms()
plt.show()
fig = length_analyzer.plot_boxplots()
plt.show()

## Paso 8 — Ingeniería de Características (sobre texto con sinónimos)

Bag of N-Grams y TF-IDF se construyen sobre `_no_stopwords_synonyms`.
La normalización previa reduce la dimensionalidad del vocabulario, lo que
produce matrices más compactas y términos TF-IDF más discriminativos.

In [ ]:
main_corpus = (
    demo_df[f"{demo_col}_no_stopwords_synonyms"]
    .replace("", pd.NA)
    .dropna()
)
print(f"Documents for vectorisation: {len(main_corpus)}\n")

bow, tfidf = build_feature_matrices(
    main_corpus,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    max_features=3000,
)

In [ ]:
display(compare_vocabularies(bow, tfidf))

In [ ]:
print("\u25B8 Top 25 terms by mean TF-IDF (synonym-normalised):\n")
display(tfidf.global_term_ranking(top_k=25))

In [ ]:
# Inspect how canonical terms now dominate
print("\u25B8 TF-IDF weight of canonical terms:\n")
for term in ["bueno", "profesor", "clase", "explicar", "aprender"]:
    result = tfidf.search_term(term)
    if result:
        print(
            f"  {term:>10}: mean={result['mean_tfidf']:.4f} | "
            f"docs={result['document_count']}"
        )
    else:
        print(f"  {term:>10}: not in vocabulary")

## Paso 9 — EDA automatizado end-to-end

`run_full_eda()` ejecuta longitudes + nube + n-gramas en una sola llamada,
sobre la columna con sinónimos, para una segunda encuesta (Calidad
Estudiantes) como demostración de escalabilidad.

In [ ]:
student_survey = "Calidad Estudiantes"
student_col = SURVEY_CONFIG[student_survey]
student_syn = f"{student_col}_no_stopwords_synonyms"

student_results = run_full_eda(
    corpus_by_survey[student_survey],
    text_columns=[student_syn],
    save_dir=str(FIGURES_DIR / "calidad_estudiantes"),
    show_plots=True,
)
print(f"\nResult keys: {list(student_results.keys())}")

## Resumen de entregables generados

Este cuaderno produce los siguientes artefactos listos para la PPT:

- **`figures/wordclouds/`** — una nube de palabras PNG por encuesta
  (5 archivos), calculadas sobre texto con sinónimos normalizados.
- **`figures/ngrams/`** — gráficos comparativos de n-gramas y top-20
  unigramas por encuesta.
- **Tablas de impacto** — reducción de vocabulario por encuesta y
  términos colapsados, como evidencia del efecto de los sinónimos.

---

## Próxima fase: modelado de tópicos con BERTopic

Con el corpus ya limpio, sin stopwords y con sinónimos normalizados,
el siguiente paso natural es el **descubrimiento no supervisado de
tópicos**. BERTopic es el candidato idóneo porque:

1. Usa *embeddings* contextuales (Sentence-Transformers) que capturan
   semántica más allá de la frecuencia.
2. Produce tópicos interpretables con términos representativos por
   tópico (via c-TF-IDF).
3. Se integra de forma natural con el pipeline actual: consume
   directamente la columna `_no_stopwords_synonyms`.

*(La implementación se entregará en la siguiente iteración.)*

## Paso 10 — Modelado de tópicos con BERTopic

Con el corpus limpio, sin stopwords y con sinónimos normalizados, el
paso final es el **descubrimiento no supervisado de tópicos**. Se usa
`Mod_Bertopic.TopicModeler`, que encapsula el pipeline completo:
embeddings multilingües → UMAP → HDBSCAN → c-TF-IDF.

> **Requisito:** `poetry add bertopic sentence-transformers umap-learn hdbscan`.
> La primera ejecución descarga el modelo de embeddings (~120 MB).
> El entrenamiento sobre ~16k respuestas puede tardar varios minutos.

In [ ]:
from Mod_Bertopic import TopicModeler

# Fit on the synonym-normalised column of the main survey
topic_corpus = (
    demo_df[f"{demo_col}_no_stopwords_synonyms"]
    .replace("", pd.NA)
    .dropna()
)
print(f"Documents for topic modelling: {len(topic_corpus)}")

modeler = TopicModeler(
    min_topic_size=20,      # broader, report-friendly topics
    random_state=42,        # reproducible
)
modeler.fit(topic_corpus)

print(f"\n\u2705 Topics discovered (excluding outliers): {modeler.topic_count()}")

### 10.1 Panorama de tópicos

El tópico `-1` agrupa los documentos atípicos (outliers) que no
encajaron en ningún cluster.

In [ ]:
display(modeler.topic_overview().head(15))

### 10.2 Términos representativos por tópico

In [ ]:
# Show representative terms for the first few real topics (skip -1)
overview = modeler.topic_overview()
real_topics = [t for t in overview["Topic"].tolist() if t != -1][:5]

for topic_id in real_topics:
    print(f"\n\u25B8 Topic {topic_id}:")
    display(modeler.topic_terms(topic_id, top_k=8))

### 10.3 Documentos representativos

In [ ]:
for topic_id in real_topics[:3]:
    print(f"\n\u25B8 Topic {topic_id} \u2014 example responses:")
    for doc in modeler.representative_docs(topic_id, n_docs=2):
        print(f"   \u2022 {doc[:100]}")

### 10.4 Visualizaciones interactivas

Estas figuras Plotly son ideales para la presentación final.

In [ ]:
# Intertopic distance map
modeler.plot_topics().show()

In [ ]:
# Top terms per topic (bar charts)
modeler.plot_barchart(top_k_topics=8).show()

In [ ]:
# Topic hierarchy
modeler.plot_hierarchy().show()

### 10.5 Persistencia del modelo

Se guarda el modelo entrenado para reutilizarlo sin reentrenar.

In [ ]:
model_path = _PROJECT_ROOT / "models" / "bertopic_evaluacion_docente"
modeler.save(model_path)

# To reload later:
#   from Mod_Bertopic import TopicModeler
#   modeler = TopicModeler.load(model_path)

---

## Cierre del pipeline

Con esto el pipeline queda completo end-to-end:

1. **Fase 1** — `limpieza.py` (limpieza + stopwords desde `stopwords.xlsx`)
   y `synonyms.py` (normalización de sinónimos).
2. **Fase 2** — `EDA.py` (longitudes, nubes por encuesta, n-gramas).
3. **Fase 3** — `Engineering.py` (BoW, TF-IDF, impacto de sinónimos).
4. **Fase 4** — `Mod_Bertopic.py` (descubrimiento de tópicos).

Todos los artefactos (`figures/wordclouds/`, `figures/ngrams/`,
`models/bertopic_*`) quedan listos para la presentación final.